In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url):
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    return data

get_json(BASE_URL + 'status.json')['service']
#get_json(BASE_URL + 'orders/page-99.json')

'Northern Star Shop API'

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url):
    records = []
    url = first_url
    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
#len(orders)

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) -> None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records, path):
    with open(path, 'w', encoding = 'utf8') as f:
        json.dump(records, f, ensure_ascii=False, indent = 2)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))


In [0]:
# 4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у
# задатку 5

def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status()               
    with open(path, 'wb') as f:               
        f.write(response.content)             
    return len(response.content)

download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))

801

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams', 'matches'
    # 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
    # 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције download_file
    # 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
    #   Након тога преименовати колоне
    #   customer_name у customer
    #   customer_city у city
    # 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
    # По потреби, pd.read_csv(...) или pd.read_parquet(...)
    # 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

def extract(raw_dir):
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir,'orders.json'))
    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns = {'customer_name' : 'customer', 'customer_city' : 'city'})

    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))

    matches_df = pd.read_parquet(RAW_DIR + '/' + 'matches.parquet')

    products_df = pd.read_csv(RAW_DIR + '/' + 'products.csv')

    teams_df = pd.read_csv(RAW_DIR + '/' + 'teams.csv')

    return {'orders' : orders_df, 'products' : products_df, 'teams' : teams_df, 'matches' : matches_df}

extract(RAW_DIR)
orders_df = extract(RAW_DIR)['orders']
products_df = extract(RAW_DIR)['products']
teams_df = extract(RAW_DIR)['teams']
matches_df = extract(RAW_DIR)['matches']

In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

def clean_products(df):
    newdf = df.copy()

    newdf['team_code'] = df['team_code'].str.strip().str.upper()
    newdf['category'] = df['category'].str.strip().str.lower()
    newdf['price'] = df['price'].str.strip().str.replace(' RSD', '', regex=False)
    newdf['price'] = pd.to_numeric(newdf['price'], errors='coerce')#.astype(float)
    newdf['active'] = df['active'].replace({'yes' : True, 'Yes' : True, 'YES' : True, 'NO' : False, 'no' : False, 'No' : False}, inplace=False)

    dups = newdf.duplicated('product_id', keep='first') 
    
    return newdf[~dups]

#clean_products(products_df)
clean_products_df = clean_products(products_df)
#display(clean_products_df)
#clean_products_df['active'].value_counts()

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame 
# order_id, customer, city, payment без размака; 
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'

def clean_orders(df):
    newdf = df.copy()

    newdf['order_id'] = df['order_id'].str.strip().str.upper()
    newdf['customer'] = df['customer'].str.strip().str.title()
    newdf['city'] = df['city'].fillna('Unknown').str.strip().str.title()
    newdf['payment'] = df['payment'].str.strip()

    return newdf

#clean_orders(orders_df)['city'].value_counts()
cleaned_orders_df = clean_orders(orders_df)
display(cleaned_orders_df)

order_id,order_date,product_id,quantity,unit_price,payment,customer,city
NSS-00001,2026-08-15,1002,1,6990.0,cash,Nikola Stojanović,Zrenjanin
NSS-00002,2026-08-15,1094,3,1290.0,card,Jovana Pavlović,Čačak
NSS-00003,2026-08-15,1041,1,6990.0,card,Sara Živković,Beograd
NSS-00004,2026-08-15,1075,1,990.0,cash,Sara Đorđević,Subotica
NSS-00005,2026-08-15,1044,1,1290.0,card,Stefan Popović,Niš
NSS-00006,2026-08-15,1078,3,1990.0,cash,Jovana Đorđević,Kraljevo
NSS-00007,2026-08-15,1070,2,1490.0,card,Petar Ilić,Sombor
NSS-00008,2026-08-15,1073,3,1490.0,cash,Nikola Jovanović,Kraljevo
NSS-00009,2026-08-15,1072,3,6990.0,cash,Milica Ilić,Kragujevac
NSS-00010,2026-08-15,1074,2,1790.0,paypal,Ana Kovačević,Beograd


In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df):
    duplicates_list = df.duplicated('order_id', keep='first')
    return df[~duplicates_list], df[duplicates_list]


unique, duplicates = remove_duplicate_orders(cleaned_orders_df)


In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа 'OK' за исправан ред, у супротном разлог
# Провере радити редом и вратити први пронађени проблем
# order_id не почиње са NSS- -> 'invalid_order'
# quantity недостаје -> 'missing_quantity'
# quantity није цео број већи од нуле -> 'invalid_quantity'
# unit_price није број већи од нуле -> 'invalid_price'
# product_id не постоји међу производима -> 'unknown_product'
# Помоћ:
# pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
# pd.isna(value) препознаје и None и NaN

def validate_order(row, product_ids):
    quantity = pd.to_numeric(row['quantity'], errors='coerce') # ne menjamo originalnu vrednost, pravimo promenljivu
    unit_price = pd.to_numeric(row['unit_price'], errors='coerce')

    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'

#display(cleaned_orders_df)
product_ids = set(products_df['product_id'])
#specific_row = cleaned_orders_df.iloc[82]
#validate_order(specific_row, product_ids)

# da validiramo ceo DataFrame:
unique.apply(lambda row: validate_order(row, product_ids), axis=1).value_counts()

OK                  1800
invalid_quantity      10
missing_quantity       8
invalid_order          8
invalid_price          8
unknown_product        6
Name: count, dtype: int64

In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame
# Типови:
# quantity -> int
# unit_price -> float
# order_date -> датум (pd.to_datetime)
# Нова колона total (quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df, products):
    new_df = df.copy()
    new_df['quantity'] = pd.to_numeric(new_df['quantity'], errors='coerce')#.astype(int)
    new_df['unit_price'] = pd.to_numeric(new_df['unit_price'], errors='coerce')#.astype(float)
    new_df['order_date'] = pd.to_datetime(new_df['order_date'], errors='coerce')

    new_df['total'] = new_df['quantity'] * new_df['unit_price']

    transformed = new_df.merge(products[['product_id', 'product_name', 'team_code','category']], on = 'product_id', how = 'left') 
    return transformed

transform_orders(unique, clean_products_df)

,order_id,order_date,product_id,quantity,unit_price,payment,customer,city,total,product_name,team_code,category
0,NSS-00001,2026-08-15,1002,1.0,6990.0,cash,Nikola Stojanović,Zrenjanin,6990.0,NPM Away Jersey,NPM,jerseys
1,NSS-00002,2026-08-15,1094,3.0,1290.0,card,Jovana Pavlović,Čačak,3870.0,BRU Cap,BRU,caps
2,NSS-00003,2026-08-15,1041,1.0,6990.0,card,Sara Živković,Beograd,6990.0,RVA Home Jersey,RVA,jerseys
3,NSS-00004,2026-08-15,1075,1.0,990.0,cash,Sara Đorđević,Subotica,990.0,LSB Mug,LSB,accessories
4,NSS-00005,2026-08-15,1044,1.0,1290.0,card,Stefan Popović,Niš,1290.0,RVA Cap,RVA,caps
...,...,...,...,...,...,...,...,...,...,...,...,...
1835,NSS-01836,2026-11-15,1046,3.0,7490.0,card,Milica Popović,Subotica,22470.0,GMW Home Jersey,GMW,jerseys
1836,NSS-01837,2026-11-15,1024,1.0,1290.0,card,Marko Kovačević,Čačak,1290.0,KLT Cap,KLT,caps
1837,NSS-01838,2026-11-15,1042,2.0,6490.0,cash,Marko Nikolić,Čačak,12980.0,RVA Away Jersey,RVA,jerseys
1838,NSS-01839,2026-11-15,1004,2.0,2290.0,cash,Jelena Živković,Zrenjanin,4580.0,NPM Cap,NPM,caps


In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason (дупликати за разлог треба да имају 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df, products):
    clean_df = clean_orders(df)
    transformed_df = transform_orders(clean_df, products)
    nondups, dups = remove_duplicate_orders(transformed_df)
    dups = dups.assign(reason = 'duplicate')
    
    product_ids = set(products['product_id'])
    validation = nondups.apply(lambda row: validate_order(row, product_ids), axis=1)
    uniques = nondups.copy()
    uniques['reason'] = validation
    ok = uniques.loc[uniques['reason'] == 'OK']
    valid = transform_orders(ok, products)

    not_ok = uniques.loc[uniques['reason'] != 'OK']
    rejected = pd.concat([not_ok, dups])

    return valid, rejected

valid, rejected = split_orders(orders_df, clean_products_df)

In [0]:
# 12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, један за домаћина и један за госта.
# Користити само утакмице са status == 'finished' (одложена нема резултат)
    # Колоне:
    # match_id
    # match_date (Датум из kickoff_ts, без времена 
    # df['kickoff_ts'].dt.normalize())
    # team_code, opponent_code
    # goals_for, goals_against
    # result ('W', 'D', 'L'), points (3, 1, 0)
# Утакмице имају team_id, а производи team_code

# codes = dict(zip(teams['team_id'], teams['code'])) # Прави речник облика 
# {1: 'NPM', 2: 'FWR', ...}
# # map користи речник да претвори id у code
# home = pd.DataFrame(
# { 
# ...
# 'team_code': played['home_team_id'].map(codes),
# 'opponent_code': played['away_team_id'].map(codes),
# ...
# })

def team_results(matches, teams):
    finished_matches = matches[matches['status'] == 'finished']
    home_matches = finished_matches.merge(teams, left_on = 'home_team_id', right_on = 'team_id', how = 'left').rename(columns = {'code' : 'team_code', 'home_goals' : 'goals_for'}).merge(teams, left_on = 'away_team_id', right_on = 'team_id', how = 'left').rename(columns = {'code' : 'opponent_code', 'away_goals' : 'goals_against'})
    away_matches = finished_matches.merge(teams, left_on = 'away_team_id', right_on = 'team_id', how = 'left').rename(columns = {'code' : 'team_code', 'home_goals' : 'goals_for'}).merge(teams, left_on = 'home_team_id', right_on = 'team_id', how = 'left').rename(columns = {'code' : 'opponent_code', 'away_goals' : 'goals_against'})
    home_matches['match_date'] = home_matches['kickoff_ts'].dt.normalize()
    away_matches['match_date'] = away_matches['kickoff_ts'].dt.normalize()
    home_matches = home_matches[['match_id', 'match_date', 'team_code', 'opponent_code', 'goals_for', 'goals_against' ]]
    away_matches = away_matches[['match_id', 'match_date', 'team_code', 'opponent_code', 'goals_for', 'goals_against' ]]
    
    all = pd.concat([home_matches, away_matches], ignore_index=True)

    # nije gotovo

    return all


team_results(matches_df, teams_df)


,match_id,match_date,team_code,opponent_code,goals_for,goals_against
0,2627001,2026-08-15,BRU,FWR,2,1
1,2627002,2026-08-15,CHS,VDC,0,1
2,2627003,2026-08-15,MBA,WFT,3,0
3,2627004,2026-08-15,NPM,IVU,1,0
4,2627005,2026-08-15,PCU,HGA,1,1
...,...,...,...,...,...,...
233,2627086,2026-10-24,NPM,KLT,1,3
234,2627087,2026-10-24,RVA,WFT,0,2
235,2627088,2026-10-25,CHS,PCU,3,1
236,2627089,2026-10-25,IVU,FWR,2,0


In [0]:
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
    path = os.path.join(processed_dir, name)
    if name.endswith('.parquet'):
    df.to_parquet(path, index=False)
    else:
    df.to_csv(path, index=False)
    written[name] = len(df)
    return written

  File <command-6604894202632602>, line 4
    path = os.path.join(processed_dir, name)
    ^
IndentationError: expected an indented block after 'for' statement on line 3


In [0]:
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    # results = team_results(data['matches'], data['teams'])
    # windows = match_windows(results)
    # outputs = {
    #     'clean_orders.parquet': valid,
    #     'rejected_orders.csv': rejected,
    #     'products.csv': products,
    #     'league_table.csv': league_table(results),
    #     'revenue_by_team.csv': revenue_by(valid.dropna(subset=
    #     ['team_code']), 'team_code'),
    #     'revenue_by_category.csv': revenue_by(valid, 'category'),
    #     'sales_after_results.csv': sales_after_results(valid, windows),
    # }
    # written = load(outputs, processed_dir)
    # summary = {
    #     'total_rows': len(data['orders']),
    #     'duplicates': int((rejected['reason'] == 'duplicate').sum()),
    #     'valid_rows': len(valid),
    #     'rejected_rows': len(rejected),
    #     'total_revenue': round(float(valid['total'].sum()), 2),
    #     'files_written': len(written) + 1,
    # }
    # pd.DataFrame(list(summary.items()), columns=['metric', 
    # 'value']).to_csv(
    # os.path.join(processed_dir, 'summary.csv'), index=False)
    # return summary
    return valid, rejected

run_pipeline(RAW_DIR, PROCESSED_DIR)

/home/spark-5c7c173d-adfe-42ef-aefa-59/.ipykernel/75/command-6604894202632618-2968888976:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newdf['active'] = df['active'].replace({'yes' : True, 'Yes' : True, 'YES' : True, 'NO' : False, 'no' : False, 'No' : False}, inplace=False)


(       order_id order_date  ...  team_code_y   category_y
 0     NSS-00001 2026-08-15  ...          NPM      jerseys
 1     NSS-00002 2026-08-15  ...          BRU         caps
 2     NSS-00003 2026-08-15  ...          RVA      jerseys
 3     NSS-00004 2026-08-15  ...          LSB  accessories
 4     NSS-00005 2026-08-15  ...          RVA         caps
 ...         ...        ...  ...          ...          ...
 1795  NSS-01836 2026-11-15  ...          GMW      jerseys
 1796  NSS-01837 2026-11-15  ...          KLT         caps
 1797  NSS-01838 2026-11-15  ...          RVA      jerseys
 1798  NSS-01839 2026-11-15  ...          NPM         caps
 1799  NSS-01840 2026-11-15  ...          RVA  accessories
 
 [1800 rows x 16 columns],
        order_id order_date  product_id  ...  team_code     category            reason
 33    NSS-00034 2026-08-16        1031  ...        BMC      jerseys  missing_quantity
 76    TST-00077 2026-08-18        1098  ...        GSR      scarves     invalid_order
 8